# 9 · Dataset & label audit (Gate Step A)

**Before** we retrain any detector we verify the training labels — they are the most likely culprit. The sentence heads (`s1`) were labelled by `label_by_reference_match` (truthful iff a gold alias is a literal **substring** of the answer, else hallucinated). On sentence-format answers that mislabels true sentences whenever the answer paraphrases / abbreviates / gives a decade-not-year / the alias list is incomplete.

This notebook (CPU-only, no model) quantifies three things:
1. **per-head held-out AUROC** on `data/claims_s1.parquet` — but measured against the *noisy* labels, so read with care;
2. the **live-demo per-detector distribution** (HalluShift flatness, SEP firing rate) from `data/demo_scores.jsonl`;
3. a **label inspection** of both classes + a hand-label worksheet for the gold-set agreement check.

All logic lives in `tools/diagnose_heads.py` so the CLI and this notebook agree.

In [ ]:
import os, sys
sys.path.insert(0, os.path.join('..', 'tools'))
import importlib, diagnose_heads as D; importlib.reload(D)
TAG = 's1'
print('audit tag =', TAG)

## 1 · Per-head held-out AUROC (against the CURRENT, noisy labels)
If a head scored ≈0.5 here it would look dead — but a head can also look *good* here purely by agreeing with the label noise (e.g. SEP fires on specific/rare tokens, and the labeller mislabels specific *true* answers as hallucinated — so they correlate). Treat these as **not trustworthy until the labels are fixed**; the real test is re-running this after Step B.

In [ ]:
D.per_head_auroc(TAG)

## 2 · Live-demo per-detector distribution
No ground truth here — just the spread of each detector over every claim sentence the demo has scored. Watch for **HalluShift flatness** (tiny std → it says ≈0.5 for everything) and **SEP firing rate** (fraction of sentences with `sep_entropy>0.5`).

In [ ]:
D.demo_distributions()

## 3 · Label inspection — eyeball the 'hallucinated' class for mislabelled TRUE sentences
The substring rule is reliable in ONE direction: a match → truthful is almost always right. It is the **non-match → hallucinated** decision that is wrong ~40% of the time. Scan the HALLUCINATED list below for sentences that are actually true. Also writes `data/label_audit_<TAG>.csv` — fill its `true_label` column by hand to measure the labeller's accuracy and (after Step B) the LLM-judge's agreement.

In [ ]:
D.label_inspection(TAG, n=20)

## Verdict & next step
- The **truthful** class is clean; the **hallucinated** class is ~40% mislabelled true sentences → labels are the primary problem.
- The per-head AUROCs (SEP ≈0.72, HalluShift ≈0.65, TSV ≈0.82) are scored against those noisy labels and are **not** trustworthy — SEP's especially, because its failure mode (firing on specificity) correlates with the labeller's failure mode.
- HalluShift is genuinely **flat** in the demo (std ≈0.09).

➡️ **Gate Step B**: re-label with the LLM-judge (`LABEL_METHOD='llm_judge'`, comparative `question + gold answer` vs the sentence), then **re-run this notebook**. The decisive question: does each head's AUROC *hold* against clean labels (genuine signal) or *drop* (it had learned the label noise)? No detector retraining until the labels pass the Step C verification.